# Uji Canny untuk kompleksitas latar

Pertanyaan: apakah edge density Canny di pinggir foto bisa membedakan latar polos dan ramai pada data kita?

Kriteria lulus ditetapkan sebelum melihat hasil: AUC ≥ 0,90 dan agreement ≥ 85%. Kalau tidak lulus, latar dilabeli manual dan Canny dilaporkan sebagai metode yang dicoba.

## Bagian 1: Menyiapkan pelabelan

Sel di bagian ini memilih 40 foto, membuat lembar foto bernomor, dan menyiapkan CSV label. Skor Canny sengaja belum dihitung supaya penilaian visual tidak terpengaruh.

In [1]:
import cv2
import numpy as np
import pandas as pd

from core.config import DATA_DIR, LOG_SOURCES, REPORTS_DIR, ROOT
from core.image import load_image
from dataset.acquisition import read_log, to_photo_table

photos = to_photo_table(read_log(LOG_SOURCES[0]))
photos = photos[photos["raw_path"].map(lambda path: (DATA_DIR / path).exists())]
photos.groupby("label")["object_id"].nunique()

label
kardus     34
kertas     27
plastik    38
Name: object_id, dtype: int64

In [2]:
SEED = 42
PAIRS_PER_CLASS = 3
SINGLES_PER_CLASS = {"plastik": 8, "kertas": 7, "kardus": 7}
LABELS_CSV = ROOT / "dataset" / "labels" / "background_sample.csv"


def sample_photos(photos: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    """Pilih foto per kelas: beberapa objek diambil atas dan miring, sisanya satu foto acak."""
    chosen = []
    for label, singles in SINGLES_PER_CLASS.items():
        objects = np.sort(photos.loc[photos["label"] == label, "object_id"].unique())
        picked = rng.choice(objects, size=PAIRS_PER_CLASS + singles, replace=False)
        for object_id in picked[:PAIRS_PER_CLASS]:
            chosen += [(object_id, "atas"), (object_id, "miring")]
        for object_id in picked[PAIRS_PER_CLASS:]:
            chosen.append((object_id, str(rng.choice(["atas", "miring"]))))
    keys = pd.DataFrame(chosen, columns=["object_id", "view"])
    return keys.merge(photos, on=["object_id", "view"]).sample(frac=1, random_state=SEED)


if LABELS_CSV.exists():
    sample = pd.read_csv(LABELS_CSV, dtype=str, keep_default_na=False)
    print("Memakai sampel yang sudah ada:", LABELS_CSV.relative_to(ROOT))
else:
    picked = sample_photos(photos, np.random.default_rng(SEED))
    sample = pd.DataFrame({"no": range(1, len(picked) + 1), "raw_path": picked["raw_path"].to_numpy(), "background": ""})
    LABELS_CSV.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(LABELS_CSV, index=False)
    print("Sampel baru disimpan:", LABELS_CSV.relative_to(ROOT))

sample.merge(photos[["raw_path", "label", "view"]], on="raw_path").groupby(["label", "view"]).size().unstack()

Sampel baru disimpan: dataset\labels\background_sample.csv


view,atas,miring
label,,
kardus,6,7
kertas,7,6
plastik,6,8


In [3]:
THUMB = 400
COLUMNS = 5
PER_SHEET = 10


def thumbnail(raw_path: str, number) -> np.ndarray:
    """Foto diperkecil ke kotak THUMB x THUMB dengan nomor di pojok kiri atas."""
    image = load_image(DATA_DIR / raw_path)
    height, width = image.shape[:2]
    scale = THUMB / max(height, width)
    image = cv2.resize(image, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
    canvas = np.full((THUMB, THUMB, 3), 255, np.uint8)
    top, left = (THUMB - image.shape[0]) // 2, (THUMB - image.shape[1]) // 2
    canvas[top : top + image.shape[0], left : left + image.shape[1]] = image
    cv2.rectangle(canvas, (0, 0), (80, 56), (255, 255, 255), -1)
    cv2.putText(canvas, str(number), (8, 44), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 0), 3)
    return cv2.copyMakeBorder(canvas, 4, 4, 4, 4, cv2.BORDER_CONSTANT, value=(180, 180, 180))


def contact_sheet(tiles: list[np.ndarray], columns: int) -> np.ndarray:
    """Susun potongan foto menjadi grid; sisa slot diisi kotak putih."""
    tiles = tiles + [np.full_like(tiles[0], 255)] * (-len(tiles) % columns)
    rows = [np.hstack(tiles[i : i + columns]) for i in range(0, len(tiles), columns)]
    return np.vstack(rows)


sheet_dir = REPORTS_DIR / "dataset"
sheet_dir.mkdir(parents=True, exist_ok=True)
for start in range(0, len(sample), PER_SHEET):
    part = sample.iloc[start : start + PER_SHEET]
    sheet = contact_sheet([thumbnail(p, n) for p, n in zip(part["raw_path"], part["no"])], COLUMNS)
    path = sheet_dir / f"background_sheet_{start // PER_SHEET + 1}.png"
    path.write_bytes(cv2.imencode(".png", sheet)[1].tobytes())
    print("disimpan:", path.relative_to(ROOT))

disimpan: reports\dataset\background_sheet_1.png
disimpan: reports\dataset\background_sheet_2.png
disimpan: reports\dataset\background_sheet_3.png
disimpan: reports\dataset\background_sheet_4.png


### Cara melabeli

1. Buka `reports/dataset/background_sheet_1.png` sampai `background_sheet_4.png`.
2. Isi kolom `background` di `dataset/labels/background_sample.csv` untuk setiap nomor.
3. Label mengikuti aturan di `docs/panduan-label-latar.md`.

Bagian 2 baru dijalankan setelah semua label terisi.